# β-VAE

---
## 目的
β-VAEを構築し，通常のVAE（`variational_autoencoder.ipynb`）と比較して，より解釈しやすい（disentangleな）潜在表現を獲得する仕組みを理解する．潜在空間の各次元を個別に動かす「Latent Traversal」により，各次元がどのような生成要因に対応しているかを可視化する．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
import numpy as np
from time import time
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## β-VAEとは
`variational_autoencoder.ipynb`のVAEは，以下の誤差関数（変分下限の負値）を最小化するように学習しました．

$$\mathcal{L}(x,z) = -\mathbb{E}_{q(z|x)}[\log p(x|z)] + D_{KL}[q(z|x)\|p(z)]$$

β-VAE[1]は，第2項のKLダイバージェンス（正則化項）に重み$\beta$を掛けた，以下の誤差関数を最小化します．

$$\mathcal{L}_{\beta}(x,z) = -\mathbb{E}_{q(z|x)}[\log p(x|z)] + \beta \cdot D_{KL}[q(z|x)\|p(z)]$$

$\beta = 1$のときは通常のVAEと一致します．$\beta > 1$とすることで，潜在変数の各次元が互いに独立な標準正規分布に近づくよう強く制約され，結果として潜在空間の各次元が「回転角度」「太さ」といった，人間が解釈しやすい独立な生成要因（disentangled representation）に対応しやすくなることが知られています．一方で，正則化を強めすぎる（$\beta$を大きくしすぎる）と，再構成の精度とのトレードオフにより，画像の復元性能は低下します．

[1] I. Higgins et al., "beta-VAE: Learning Basic Visual Concepts with a Constrained Variational Framework," ICLR, 2017.

## ネットワークの構築
ネットワーク構造は，`variational_autoencoder.ipynb`のVAEと全く同じ（Encoder: `784 → 256 → 100 → {μ, logσ}`，Decoder: `潜在変数 → 100 → 256 → 784`）です．構造の詳細は`variational_autoencoder.ipynb`を参照してください．β-VAEは，ネットワーク構造ではなく**誤差関数のKLダイバージェンス項に対する重み付け**が異なる手法です．

潜在空間の各次元と生成要因の対応関係を確認しやすくするため，本ノートブックでは潜在変数の次元数を`10`とします（`variational_autoencoder.ipynb`では可視化の都合上`2`次元としていました）．

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=10):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(28 * 28, 256), nn.ReLU(inplace=True),
            nn.Linear(256, 100), nn.ReLU(inplace=True),
        )
        self.l_mu = nn.Linear(100, latent_dim)
        self.l_logvar = nn.Linear(100, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 100), nn.ReLU(inplace=True),
            nn.Linear(100, 256), nn.ReLU(inplace=True),
            nn.Linear(256, 28 * 28),  # Sigmoidは適用しない（誤差関数側でまとめて適用する）
        )

    def reparameterize(self, mu, logvar):
        std = logvar.mul(0.5).exp()
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu = self.l_mu(h)
        logvar = self.l_logvar(h)
        latent = self.reparameterize(mu, logvar)
        out = self.decoder(latent)
        return out, mu, logvar

## データセット，ネットワーク，最適化関数の設定
データセットにはMNISTを使用します．最適化手法にはAdam optimizer（学習率$10^{-3}$）を使用します．

In [ ]:
mnist_data = datasets.MNIST(root='./data', train=True, transform=transforms.ToTensor(), download=True)
train_loader = DataLoader(mnist_data, batch_size=100, shuffle=True)

latent_dim = 10
model = VAE(latent_dim=latent_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## 誤差関数の設定
`variational_autoencoder.ipynb`と同じBCE lossとKLダイバージェンスを用いますが，KLダイバージェンス項に重み`beta`を掛ける点が異なります．

In [ ]:
beta = 4.0


def loss_function(tilde_x, x, mu, logvar, bce, beta):
    reconstruction_loss = bce(tilde_x.view(-1, 784), x.view(-1, 784))
    kl_div = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return reconstruction_loss + beta * kl_div


bce = nn.BCEWithLogitsLoss(reduction='sum').to(device)

## 学習
学習エポック数を10として学習します．

In [ ]:
epoch_num = 10

model.train()
start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0
    for x, _ in train_loader:
        x = x.to(device)

        tilde_x, mu, logvar = model(x.view(x.size(0), -1))
        loss = loss_function(tilde_x, x, mu, logvar, bce, beta)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()

    print(f'epoch: {epoch}, mean loss: {sum_loss / len(train_loader):.4f}, elapsed_time: {time() - start:.4f}')

## 学習済みモデルを用いた画像の復元
評価用データからランダムに画像をサンプルし，β-VAEによる再構成結果を確認します．`variational_autoencoder.ipynb`と同様，Decoderの出力は生のロジットであるため，画像として表示する前に`torch.sigmoid`を適用します．

In [ ]:
mnist_testdata = datasets.MNIST(root='./data', train=False, transform=transforms.ToTensor())
test_loader = DataLoader(mnist_testdata, batch_size=10, shuffle=True)

model.eval()
with torch.no_grad():
    x, _ = next(iter(test_loader))
    x = x.to(device)
    tilde_x, mu, logvar = model(x.view(x.size(0), -1))
    tilde_x = torch.sigmoid(tilde_x)

x = x.cpu().view(-1, 28, 28).numpy()
tilde_x = tilde_x.cpu().view(-1, 28, 28).numpy()

fig, axes = plt.subplots(2, 10, figsize=(14, 2.8))
for i in range(10):
    axes[0, i].imshow(x[i], cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(tilde_x[i], cmap='gray'); axes[1, i].axis('off')
fig.suptitle('input (top) / reconstruction (bottom)')
plt.show()

## Latent Traversal（潜在空間の次元ごとの可視化）
潜在変数が本当にdisentangleな表現を獲得できているかを確認するため，1枚の画像をEncodeして得た潜在変数$\mu$を基準に，**1つの次元だけ**を$-3$から$3$まで変化させ，他の次元は固定したままDecodeします．これを全ての次元について行うことで，各次元がどのような見た目の変化（生成要因）に対応しているかを確認できます．解釈しやすい表現が獲得できていれば，例えば「ある次元は数字の太さ，別の次元は傾き」のように，1つの次元の変化が1種類の見た目の変化にのみ対応するはずです．

In [ ]:
model.eval()
image = mnist_testdata[0][0]  # 基準となる1枚の画像
with torch.no_grad():
    h = model.encoder(image.view(1, -1).to(device))
    base_mu = model.l_mu(h)  # この画像の潜在変数（基準点）

n_steps = 10
traversal_range = np.linspace(-3, 3, n_steps)

fig, axes = plt.subplots(latent_dim, n_steps, figsize=(1.4 * n_steps, 1.4 * latent_dim))
with torch.no_grad():
    for dim in range(latent_dim):
        z = base_mu.repeat(n_steps, 1).clone()
        z[:, dim] = torch.tensor(traversal_range, dtype=torch.float32).to(device)  # 1次元だけ動かす

        output = torch.sigmoid(model.decoder(z)).view(-1, 28, 28).cpu().numpy()
        for step in range(n_steps):
            axes[dim, step].imshow(output[step], cmap='gray')
            axes[dim, step].axis('off')
        axes[dim, 0].set_ylabel(f'z[{dim}]', rotation=0, labelpad=20)

plt.tight_layout()
plt.show()

## 課題

1. `beta`の値を`1`（通常のVAEと同じ）・`4`・`25`などに変更して学習し，Latent Traversalの結果や再構成画像の質がどのように変化するか比較してください．
2. `latent_dim`を変更し，disentangleな表現が得られやすい次元数について考察してください．
3. `variational_autoencoder.ipynb`のVAE（`latent_dim=2`）で同様にLatent Traversalを行い，本ノートブックのβ-VAEと比較してください．

## 参考文献
[1] I. Higgins, L. Matthey, A. Pal, C. Burgess, X. Glorot, M. Botvinick, S. Mohamed, A. Lerchner, "beta-VAE: Learning Basic Visual Concepts with a Constrained Variational Framework," ICLR, 2017.\
[2] Diederik P. Kingma and Max Welling, "Auto-Encoding Variational Bayes," ICLR, 2014.